# Laboratorio 2 — Índice Invertido

Un **índice invertido** es una estructura de datos que mapea cada palabra del corpus
directamente a la lista de documentos donde aparece.

En lugar de recorrer cada documento buscando una palabra (búsqueda lineal O(n)),
el índice permite hacer lookup directo en O(1).

Estructura:
  ``{ "palabra" : ["doc1.txt", "doc3.txt", "doc7.txt"] }``

In [1]:
import os
import time
import pandas as pd

import sys

sys.path.append("../../")
import libreria as ir

## Sección 1 — Carga del corpus

In [ ]:
# Carga de un solo archivo para inspección inicial
corpus_path = "../corpus/11_Alices Adventures in Wonderland.txt"

file_alice = ir.load_file(corpus_path)

print(f"Caracteres cargados : {len(file_alice)}")
print(f"Primeros 200 caracteres:\n{file_alice[:200]}")

In [ ]:
# Listar todos los archivos del corpus
corpus_dir = "../corpus/"
files = [f for f in os.listdir(corpus_dir) if f.endswith(".txt")]

print(f"Total de archivos en el corpus: {len(files)}")
print(f"Primeros 5: {files[:5]}")

In [ ]:
# Carga completa con la función de la librería
corpus = ir.load_corpus("../corpus/")

## Sección 2 — Exploración previa: vocabulario de un archivo

Antes de construir el índice, exploramos cómo se ve el vocabulario
de un solo documento: cuántas palabras tiene en total y cuántas son únicas.

In [9]:
# Tomamos un archivo del corpus como ejemplo
archivo_ejemplo = files[0]  # Puedes cambiar el índice para inspeccionar otro archivo
print(f"Archivo: {archivo_ejemplo}")

with open(os.path.join(corpus_dir, archivo_ejemplo), "r", encoding="utf-8") as f:
    contenido = f.read()

palabras    = contenido.split()
vocabulario = set(palabras)

print(f"Total de palabras  : {len(palabras)}")
print(f"Palabras únicas    : {len(vocabulario)}")

Archivo: 100_The Complete Works of William Shakespeare.txt
Total de palabras  : 966506
Palabras únicas    : 71598


### ⚠️ Observación importante: sin normalizar, el vocabulario se infla

`split()` separa por espacios pero NO limpia puntuación ni mayúsculas.
Esto hace que una misma palabra cuente como varias entradas distintas:

  * "people"   → 1 entrada
  * "people,"  → 1 entrada distinta
  * "people."  → 1 entrada distinta
  * "People"   → 1 entrada distinta

Esto infla el vocabulario y ensucia el índice.
La normalización se abordará en el siguiente laboratorio.

## Sección 3 — Construcción del índice invertido

El índice es un diccionario donde:
- **Clave**  : cada palabra única del corpus
- **Valor**  : lista de archivos donde esa palabra aparece (posting list)

  ``{ "Alice"  : ["11_Alices Adventures.txt", "98_A Tale of Two Cities.txt"],``

  ``"rabbit" : ["11_Alices Adventures.txt"] }``

Lo construimos en dos pasos: primero para un solo archivo (concepto),
luego para todo el corpus.

### 3.1 — Índice para un solo archivo

In [10]:
# Índice para un único archivo — para entender la estructura base
# Reutilizamos el archivo_ejemplo y vocabulario de la Sección 2

invert_index = {}

for palabra in vocabulario:
    invert_index[palabra] = [archivo_ejemplo]  # lista con el único archivo

# Inspección: primeras 10 entradas
print(f"Entradas en el índice: {len(invert_index)}")
print("\nMuestra de 10 entradas:")
for palabra, archivos in list(invert_index.items())[:10]:
    print(f"  '{palabra}' → {archivos}")

Entradas en el índice: 71598

Muestra de 10 entradas:
  'shillings' → ['100_The Complete Works of William Shakespeare.txt']
  'fists' → ['100_The Complete Works of William Shakespeare.txt']
  'seemly' → ['100_The Complete Works of William Shakespeare.txt']
  'yourselves?' → ['100_The Complete Works of William Shakespeare.txt']
  'pickaxes' → ['100_The Complete Works of William Shakespeare.txt']
  'Supported' → ['100_The Complete Works of William Shakespeare.txt']
  'triumphers' → ['100_The Complete Works of William Shakespeare.txt']
  'enfeebled' → ['100_The Complete Works of William Shakespeare.txt']
  'widower’s' → ['100_The Complete Works of William Shakespeare.txt']
  'THALIARD,' → ['100_The Complete Works of William Shakespeare.txt']


### 3.2 — Índice para todo el corpus

Ahora iteramos sobre todos los archivos.
Por cada palabra del vocabulario de cada documento:
- Si ya está en el índice → agregamos el archivo a su lista
- Si no está             → creamos la entrada con ese archivo

In [11]:
# Construcción del índice invertido completo sobre los 100 archivos
invert_index = {}

for archivo in files:
    with open(os.path.join(corpus_dir, archivo), "r", encoding="utf-8") as f:
        contenido = f.read()

    vocabulario_archivo = set(contenido.split())  # palabras únicas del archivo

    for palabra in vocabulario_archivo:
        if palabra in invert_index:
            invert_index[palabra].append(archivo)  # ya existe → agregar
        else:
            invert_index[palabra] = [archivo]      # nueva entrada

    print(f"✔ {archivo} — vocabulario: {len(vocabulario_archivo)} palabras")

print(f"\nÍndice construido: {len(invert_index)} entradas únicas en total")

✔ 100_The Complete Works of William Shakespeare.txt — vocabulario: 71598 palabras
✔ 1080_A Modest Proposal.txt — vocabulario: 2070 palabras
✔ 11030_Incidents in the Life of a Slave Girl.txt — vocabulario: 11794 palabras
✔ 1184_The Count of Monte Cristo.txt — vocabulario: 40033 palabras
✔ 11_Alices Adventures in Wonderland.txt — vocabulario: 5974 palabras
✔ 120_Treasure Island.txt — vocabulario: 11943 palabras
✔ 1232_The Prince.txt — vocabulario: 8482 palabras
✔ 1259_Twenty Years After.txt — vocabulario: 25686 palabras
✔ 1260_Jane Eyre.txt — vocabulario: 27928 palabras
✔ 13188_Putnams Word Book.txt — vocabulario: 44232 palabras
✔ 1342_Pride and Prejudice.txt — vocabulario: 14703 palabras
✔ 1400_Great Expectations.txt — vocabulario: 22270 palabras
✔ 145_Middlemarch.txt — vocabulario: 32192 palabras
✔ 1513_Romeo and Juliet.txt — vocabulario: 6958 palabras
✔ 15399_The Interesting Narrative of the Life of Olaudah Equiano.txt — vocabulario: 12390 palabras
✔ 161_Sense and Sensibility.txt — vo

## Sección 5 — Exploración del índice

Con el índice construido, lo exploramos desde distintos ángulos:
cuántas entradas tiene, qué palabras son compartidas entre documentos,
y por qué el tamaño no es la suma de los vocabularios individuales.

### 5.1 — Tamaño total del índice

In [12]:
# Total de palabras únicas registradas en el índice
print(f"Palabras únicas en el índice: {len(invert_index)}")

# Para contraste: suma de vocabularios individuales (sin unir)
suma_vocabularios = sum(len(set(open(os.path.join(corpus_dir, f), encoding="utf-8").read().split())) for f in files)
print(f"Suma de vocabularios individuales: {suma_vocabularios}")
print(f"Diferencia (palabras compartidas entre docs): {suma_vocabularios - len(invert_index)}")

Palabras únicas en el índice: 774028
Suma de vocabularios individuales: 2122210
Diferencia (palabras compartidas entre docs): 1348182


### 5.2 — Palabras compartidas entre más de un documento

Estas son las palabras cuya posting list tiene más de un archivo.
Son las más "valiosas" para la búsqueda: conectan documentos distintos.

In [13]:
# Palabras que aparecen en más de un archivo
compartidas = {palabra: archivos for palabra, archivos in invert_index.items() if len(archivos) > 1}

print(f"Palabras que aparecen en más de 1 archivo: {len(compartidas)}")
print("\nMuestra de 10:")
for palabra, archivos in list(compartidas.items())[:10]:
    print(f"  '{palabra}' → {len(archivos)} archivos")

Palabras que aparecen en más de 1 archivo: 179882

Muestra de 10:
  'shillings' → 30 archivos
  'fists' → 25 archivos
  'seemly' → 15 archivos
  'yourselves?' → 3 archivos
  'Supported' → 4 archivos
  'triumphers' → 2 archivos
  'enfeebled' → 14 archivos
  'shilling;' → 10 archivos
  'Today,' → 6 archivos
  'temporary' → 56 archivos


### 5.3 — ¿Por qué el índice no es la suma de los vocabularios?

Si el corpus tiene 100 archivos, cada uno con su vocabulario propio,
la cantidad de entradas en el índice NO es la suma de todos esos vocabularios.

La razón es la **intersección**: las palabras comunes entre documentos
se registran UNA sola vez en el índice (como clave), con todos sus
documentos en la posting list.

La fórmula para la unión de conjuntos es:

  |A ∪ B| = |A| + |B| − |A ∩ B|

Generalizado a N documentos:

  |índice| ≤ Σ|vocabulario_i|

Cuantas más palabras comparten los documentos, más pequeño
es el índice respecto a esa suma.

### 5.4 — Visualización con pandas

In [15]:
# Vista tabular del índice completo
df = pd.DataFrame(
    [(palabra, archivos, len(archivos)) for palabra, archivos in invert_index.items()],
    columns=["Palabra", "Posting List", "# Documentos"]
)

# Ordenado por número de documentos descendente
df = df.sort_values("# Documentos", ascending=False).reset_index(drop=True)

print(f"Total de filas: {len(df)}")
df.sample(10)  # Muestra aleatoria de 10 filas para inspección

Total de filas: 774028


,Palabra,Posting List,# Documentos
721474,Muff’s,[74_The Adventures of Tom Sawyer.txt],1
293586,"saint.""",[37106_Little Women.txt],1
477145,"askers,",[51252_The Book of the Thousand Nights and a N...,1
158673,"Hallo,","[1184_The Count of Monte Cristo.txt, 28054_The...",2
267818,Parting.,[3011_The Lady of the Lake.txt],1
560121,[A13-285],[58031_Report Warren Commission JFK.txt],1
175585,co-heirs;,"[1952_The Yellow Wallpaper.txt, 50559_Works of...",2
573210,carton.[C4-210],[58031_Report Warren Commission JFK.txt],1
266589,"“Francis,”",[100_The Complete Works of William Shakespeare...,1
461524,[5107],[49008_Works of Shakespeare Cambridge Vol 8.txt],1


## Sección 6 — Búsqueda en el índice

Con el índice construido, buscar una palabra es un lookup directo al diccionario.
Complejidad O(1) — sin importar el tamaño del corpus.

### 6.1 — Función de búsqueda

In [16]:
# Búsqueda por palabra en el índice
# Retorna la posting list, o lista vacía si no existe

def buscar_palabra(indice, palabra):
    return indice.get(palabra, [])

In [17]:
# Pruebas: palabras comunes, raras y ausentes
queries = ["the", "Alice", "Manuel", "Frankenstein", "xzqwerty"]

for q in queries:
    resultado = buscar_palabra(invert_index, q)
    if resultado:
        print(f"'{q}' → encontrada en {len(resultado)} archivo(s): {resultado[:3]}{'...' if len(resultado) > 3 else ''}")
    else:
        print(f"'{q}' → no encontrada en el índice")

'the' → encontrada en 100 archivo(s): ['100_The Complete Works of William Shakespeare.txt', '1080_A Modest Proposal.txt', '11030_Incidents in the Life of a Slave Girl.txt']...
'Alice' → encontrada en 15 archivo(s): ['100_The Complete Works of William Shakespeare.txt', '11_Alices Adventures in Wonderland.txt', '1260_Jane Eyre.txt']...
'Manuel' → encontrada en 4 archivo(s): ['39647_The Spanish American Reader.txt', '58031_Report Warren Commission JFK.txt', '6761_The Adventures of Ferdinand Count Fathom.txt']...
'Frankenstein' → encontrada en 2 archivo(s): ['71046_Modern English Biography Volume 2.txt', '84_Frankenstein.txt']
'xzqwerty' → no encontrada en el índice


### 6.3 — Búsqueda masiva + medición de tiempo

Buscamos TODAS las palabras del índice y medimos cuánto tarda.
Esto es el equivalente al `search_bulk` del lab 1, pero usando el índice.

In [18]:
# Búsqueda masiva: todas las palabras del índice
todas_las_palabras = list(invert_index.keys())

start = time.time()

resultados_masivos = {}
for palabra in todas_las_palabras:
    resultados_masivos[palabra] = buscar_palabra(invert_index, palabra)

end = time.time()

elapsed_ms = (end - start) * 1000

print(f"Palabras buscadas  : {len(todas_las_palabras)}")
print(f"Tiempo total       : {elapsed_ms:.4f} ms")
print(f"Tiempo promedio    : {elapsed_ms / len(todas_las_palabras):.6f} ms por palabra")

Palabras buscadas  : 774028
Tiempo total       : 773.7482 ms
Tiempo promedio    : 0.001000 ms por palabra


### Comparación rápida con el lab 1

En el lab 1, `search_bulk` buscaba todas las palabras de Alice (~5,974)
recorriendo linealmente los 100 archivos → tardaba varios minutos.

Aquí buscamos todas las palabras del índice completo (~774k entradas)
y tarda milisegundos. La diferencia es la estructura: el índice ya sabe
exactamente en qué documentos está cada palabra.

## Sección 8 — Comparación: índice invertido vs. búsqueda lineal

En el lab 1 medimos el tiempo de búsqueda lineal sobre el corpus.
Aquí repetimos la misma búsqueda pero usando el índice invertido
para ver el contraste real en tiempos.

### 8.1 — Búsqueda lineal (lab 1) vs. búsqueda en índice

Usamos las mismas palabras del vocabulario de Alice como queries,
igual que en el lab 1, para que la comparación sea justa.

In [19]:
# Queries: palabras únicas del libro de Alice (mismo criterio que lab 1)
palabras_alice = set(corpus["11_Alices Adventures in Wonderland.txt"].split())

print(f"Total de queries (vocabulario de Alice): {len(palabras_alice)}")

Total de queries (vocabulario de Alice): 5974


In [20]:
# --- Búsqueda LINEAL (lab 1) ---
resultado_lineal = ir.search_bulk(corpus, palabras_alice)

tiempo_lineal_ms  = resultado_lineal["total_time_ms"]
tiempo_lineal_avg = resultado_lineal["avg_time_ms"]

print(f"[Lineal] Tiempo total   : {tiempo_lineal_ms:.2f} ms")
print(f"[Lineal] Tiempo promedio: {tiempo_lineal_avg:.4f} ms por query")

[Lineal] Tiempo total   : 267348.51 ms
[Lineal] Tiempo promedio: 44.7520 ms por query


In [21]:
# --- Búsqueda en ÍNDICE INVERTIDO ---
start = time.time()

for palabra in palabras_alice:
    buscar_palabra(invert_index, palabra)

end = time.time()

tiempo_indice_ms  = (end - start) * 1000
tiempo_indice_avg = tiempo_indice_ms / len(palabras_alice)

print(f"[Índice] Tiempo total   : {tiempo_indice_ms:.2f} ms")
print(f"[Índice] Tiempo promedio: {tiempo_indice_avg:.6f} ms por query")

[Índice] Tiempo total   : 329.15 ms
[Índice] Tiempo promedio: 0.055096 ms por query


### 8.2 — Tabla resumen

In [22]:
# Tabla comparativa
speedup = tiempo_lineal_ms / tiempo_indice_ms if tiempo_indice_ms > 0 else float("inf")

resumen = pd.DataFrame([
    {
        "Método"           : "Búsqueda lineal (lab 1)",
        "Queries"          : len(palabras_alice),
        "Tiempo total (ms)": round(tiempo_lineal_ms, 2),
        "Avg por query (ms)": round(tiempo_lineal_avg, 4),
    },
    {
        "Método"           : "Índice invertido (lab 2)",
        "Queries"          : len(palabras_alice),
        "Tiempo total (ms)": round(tiempo_indice_ms, 2),
        "Avg por query (ms)": round(tiempo_indice_avg, 6),
    }
])

print(f"El índice invertido es {speedup:.0f}x más rápido en búsqueda\n")
resumen

El índice invertido es 812x más rápido en búsqueda



,Método,Queries,Tiempo total (ms),Avg por query (ms)
0,Búsqueda lineal (lab 1),5974,267348.51,44.752000
1,Índice invertido (lab 2),5974,329.15,0.055096
